# Training Script Without Bells and Whistles

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader

from src.data_utils import setup_transforms
from src.animate_loader import animate_loader
from src.config_utils import setup_config_variables
from src.training_utils import run_epoch
from src.logging_utils import report_progress, log_loss
from src.model_utils import save_model
from lazy_dataset import LazyDataset

from modified_resnet import ModifiedResnet

config = setup_config_variables()

In [ ]:
model = ModifiedResnet(in_channels=config["MODEL_DIMS"])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config["LEARNING_RATE"], weight_decay=config["WEIGHT_DECAY"])

In [ ]:
train_transform, train_target_transform = setup_transforms(augment_data=True, use_aps=config["USE_APS"], use_dvs=config["USE_DVS"])
test_transform, test_target_transform = setup_transforms(augment_data=False, use_aps=config["USE_APS"], use_dvs=config["USE_DVS"])

train_dataset = LazyDataset(config["TRAIN_DIR"], transform=train_transform, target_transform=train_target_transform)
test_dataset = LazyDataset(config["TEST_DIR"], transform=test_transform, target_transform=test_target_transform)

train_loader = DataLoader(train_dataset, batch_size=config["BATCH_SIZE"], shuffle=True, num_workers=4, pin_memory=True, prefetch_factor=5)
report_progress(config["GLOBAL_RANK"], f"Train loader has {len(train_loader)} batches, equalling {len(train_loader) * config['BATCH_SIZE']} frames.")
test_loader = DataLoader(test_dataset, batch_size=config["BATCH_SIZE"], shuffle=True, num_workers=4, pin_memory=True, prefetch_factor=5)
report_progress(config["GLOBAL_RANK"], f"Test loader has {len(test_loader)} batches, equalling {len(test_loader) * config['BATCH_SIZE']} frames.")

In [ ]:
starting_epoch = 0
best_test_loss = np.inf
last_model_dir = f"{config['MODEL_SAVE_DIR']}/last"
best_model_dir = f"{config['MODEL_SAVE_DIR']}/best"

if config["ANIMATE_LOADER"] and config["MODEL_DIMS"] == 2:
        animate_loader(train_loader, save_path=config["IMAGES_DIR"])

for epoch in range(starting_epoch, config["EPOCHS"]):
        report_progress("Beginning Epoch", epoch=epoch, total_epochs=config["EPOCHS"])
        epoch_train_loss = run_epoch(model, train_loader, optimizer, criterion, "TRAIN")
        epoch_test_loss = run_epoch(model, train_loader, optimizer, criterion, "TEST")
        is_new_best = epoch_test_loss < best_test_loss
        report_progress(f"Train Loss (RMSE in degs of steering): {180 * (epoch_train_loss ** 0.5): .4f}", epoch=epoch, total_epochs=config["EPOCHS"])
        report_progress(f"Test Loss (RMSE in degs of steering): {180 * (epoch_test_loss ** 0.5): .4f}", epoch=epoch, total_epochs=config["EPOCHS"])
        if is_new_best:
                report_progress(f"New best model found, prev. best test loss {180 * (best_test_loss ** 0.5)}", epoch=epoch, total_epochs=config["EPOCHS"])
        log_loss(epoch, epoch_train_loss, epoch_test_loss, config["MODEL_SAVE_DIR"])
        save_model(last_model_dir, model, optimizer, epoch, epoch_test_loss, best_test_loss, clean_old=True)